# PersonaPlex + IMTalker Live Avatar Server with Emotion — RunPod RTX 5090 Deployment

This notebook deploys and launches the **live PersonaPlex + IMTalker AH AudioPace
server** described in `live.md` (cross-checked against `live_working.md` and
`winner_latest_1.md`), the authoritative deployment documents for this stack:

```
PersonaPlex assistant audio + hidden states (12.5 Hz)
-> assistant-output RMS gate (silence Helium seed while not replying)
-> 0.96s lookahead adapter (unitalk_last_layer, 12 hidden steps + 6 future steps)
-> IMTalker generator with static-head LoRA (2h "live_winner" checkpoint)
-> emotion adapter: one session-pinned emotion (happy/sad/angry/surprise/fear)
-> IMTalker FP32 renderer
-> separate Opus audio (/ws/conversation) + JPEG video (/ws/video) websockets
```

`run_live.sh` is a thin wrapper that always launches the **AH AudioPace**
winner (`IMTalker/start_typeah_audio_pace_varm3.sh` ->
`IMTalker/start_winner_live.sh` -> `liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py`),
which fixed an intermittent assistant-audio speed-up after renderer stalls
relative to the earlier AJ NetworkIso server.

It performs, in order:

1. Validates / clones the `speech2avatar` project structure (`IMTalker/`, `personaplex/`,
   `scripts/download_live_assets.sh`, `run_live.sh`, `live.md`, and the AH-winner
   files that must be present in the clone: `start_winner_live.sh`,
   `start_typeah_audio_pace_varm3.sh`, the `AHAudioPace.py` server, the
   `RB_Robert_System_Prompt_full.txt` prompt, and the `_aj_nodrop` HTML page)
   and re-asserts `+x` on every shell script in the checkout.
2. Installs system packages, creates the `python3.11` virtualenv, installs the
   **pinned** `torch==2.8.0+cu128` stack, and all IMTalker / PersonaPlex Python
   dependencies — without ever downgrading PyTorch.
3. Authenticates to Hugging Face from a secure token prompt (never hardcoded).
   Note: `nvidia/personaplex-7b-v1` is a **gated** model repo — accept its terms
   on Hugging Face with the same account before running this notebook.
4. Downloads and **checksum-verifies** every runtime asset: the lookahead/RMS
   adapter, the silence-Helium seed, the current 2-hour static-head LoRA, the
   cached blink motion (currently unused by the AH default launch but still
   downloaded), the base IMTalker checkpoints, the PersonaPlex bnb4 weights,
   the PersonaPlex Mimi/tokenizer files, and the `VARM3` voice prompt.
5. Installs the PersonaPlex/Moshi package with `--no-deps`.
6. Runs full GPU/CUDA/checkpoint/port validation.
7. Launches the live server by invoking the **actual `run_live.sh`** (never
   reimplementing its flags), waits for the documented healthy log markers, and
   confirms the service is reachable on `0.0.0.0:{PORT}` (default `8998`) —
   **`PORT` must match an HTTP port your RunPod pod template actually exposes**,
   or the public proxy URL will never reach it even though the server is healthy.
8. Provides operational cells: log tailing, full diagnostics, and a safe stop switch.

**Emotion version:** this variant integrates the emotion adapter trained by
`train_emotion_lora.ipynb`. Select one of five emotions (`happy`, `sad`, `angry`,
`surprise`, `fear`) in the **Emotion parameters** cell; the avatar keeps that
expression for the whole conversation. Set `EMOTION = "none"` for the original
emotion-free behavior. Step 10.5 patches the generator + live server
(idempotent) and validates the trained checkpoint before launch.

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8 base image).
> Edit the **Parameters** cell below before running — in particular `PROJECT_ROOT`,
> `PORT` (must match your pod's exposed HTTP port), and, on a brand-new pod with
> no existing clone, `GIT_REPO_URL`.


In [ ]:
!git clone https://github.com/MoshiHead/-n_IMTalker_s_Personaplex_v2_2_aug_demo_code_add_time_nd_compress_log_v2_remove_search.git speech2avatar

## Parameters

Edit these before running. Leaving an override blank means "use the default
baked into `run_live.sh`" — this notebook never edits that script, it only sets
environment variables that the script already reads.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar checkout, it is used
# as-is. Otherwise, if GIT_REPO_URL is set, the repo is cloned into PROJECT_ROOT.
PROJECT_ROOT = "/workspace/speech2avatar"
GIT_REPO_URL = "https://github.com/nash-raf/speech2avatar.git"  # leave blank if already cloned (e.g. by the cell above)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
VENV_DIR = "/workspace/preprocess_5090"

# --- Service ------------------------------------------------------------------
# PORT must exactly match an "HTTP Port" / "TCP Port" exposed in your RunPod pod
# template. RunPod's proxy (https://<pod-id>-<port>.proxy.runpod.net/) only
# forwards traffic for ports that were actually exposed when the pod was
# created/edited — if the server binds a port your pod doesn't expose, the
# proxy URL will hang or refuse the connection even though the server is
# healthy. Check your pod's "Connect" -> "HTTP Service" list on the RunPod
# dashboard and set PORT to one of those values (commonly 8998 or 8999 for
# this stack; AH's own standalone default is 8998, run_live.sh's own default
# is 8999 if PORT is left unset).
HOST = "0.0.0.0"  # informational only: start_winner_live.sh always binds 0.0.0.0
PORT = 8998

# RunPod sets RUNPOD_POD_ID automatically inside every pod; used only to print
# the exact public proxy URL below. Override manually if it's not auto-detected
# (e.g. running this notebook outside of a RunPod pod).
RUNPOD_POD_ID = os.environ.get("RUNPOD_POD_ID", "")
PUBLIC_URL = f"https://{RUNPOD_POD_ID}-{PORT}.proxy.runpod.net/" if RUNPOD_POD_ID else None

# --- Hugging Face sources (must match live.md / live_working.md / winner_latest_1.md) ---
# Live lookahead/RMS adapter + silence-Helium seed (both in the same dataset repo).
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_lookahead_rms_adapter/checkpoints/personaplex_lookahead096_future048_rms50_adapter.pt"
HF_SILENCE_FILE = "personaplex_lookahead_rms_adapter/stats/silence_helium_mean.pt"

# Current AH static-head with-audio LoRA (2-hour "live_winner" checkpoint).
HF_LORA_REPO = "niloy629/hdtf_preprocess"
HF_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
HF_LORA_SHA256 = "f34de1bd748857bc802102da578046bb10bd5f664460607d9c785ff35922e52f"

# Cached blink motion-map (downloaded by download_live_assets.sh; not wired into
# the current AH/AJ default launch, kept here so the asset is still verified).
HF_BLINK_REPO = "niloy629/hdtf_preprocess"
HF_BLINK_FILE = "lora/3robert_audio3_ditto_static_motion.pt"
HF_BLINK_SHA256 = "e29a41ff004b228d7efee15cad0f32f4d4bc5466563709e2ba78b158d4e340bb"

HF_IMTALKER_REPO = "cbsjtu01/IMTalker"
HF_IMTALKER_FILES = [
    "config.yaml",
    "renderer.ckpt",
    "generator.ckpt",
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]

HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"

# Gated repo: accept the terms for nvidia/personaplex-7b-v1 on Hugging Face with
# the same account as HF_TOKEN before running Step 8, or these downloads 403.
HF_PERSONAPLEX_MIMI_REPO = "nvidia/personaplex-7b-v1"
HF_PERSONAPLEX_MIMI_FILES = [
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "tokenizer_spm_32k_3.model",
]
HF_VOICES_FILE = "voices.tgz"  # extracted to PERSONAPLEX_BNB4_DIR/voices/; VARM3.pt is the live default

EXPECTED_TORCH_VERSION = "2.8.0+cu128"

# --- Optional run_live.sh overrides (leave "" to use the script's own default) ---
REF_PATH = ""            # e.g. f"{PROJECT_ROOT}/IMTalker/assets/source_5.png"
A_CFG_SCALE = ""         # e.g. "1.13" (AH default) or "1.15" (AJ default)
NFE = ""                 # e.g. "5" (script default)
PROMPT_FILE = ""         # e.g. f"{PROJECT_ROOT}/IMTalker/prompts/RB_Robert_System_Prompt_full.txt" (script default)
VOICE_PROMPT = ""        # e.g. "VARM3.pt" (script default; use VARM3 unless explicitly told otherwise)
VOICE_PROMPT_DIR = ""    # e.g. "/workspace/voices"
LORA_GENERATOR_PATH = "" # e.g. path to an alternate static-head LoRA checkpoint
DISABLE_LORA = ""        # e.g. "1" to compare the base generator without LoRA
JITTER_BUFFER_SEC = ""   # e.g. "1.0" (script default) seconds of end-to-end A/V jitter buffer; raise to 2-3 for max smoothness


# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5

# --- Derived paths (must match download_live_assets.sh / run_live.sh) -------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
PERSONAPLEX_DIR = f"{PROJECT_ROOT}/personaplex"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
IMTALKER_CKPT_DIR = f"{IMTALKER_DIR}/checkpoints"
ADAPTER_PATH = f"{CHECKPOINT_DIR}/{HF_ADAPTER_FILE}"
SILENCE_HELIUM_PATH = f"{CHECKPOINT_DIR}/{HF_SILENCE_FILE}"
LORA_PATH = f"{CHECKPOINT_DIR}/{HF_LORA_FILE}"
BLINK_MOTION_PATH = f"{CHECKPOINT_DIR}/{HF_BLINK_FILE}"
PERSONAPLEX_WEIGHT_PATH = f"{PERSONAPLEX_BNB4_DIR}/model_bnb_4bit.pt"
VOICE_PROMPT_PATH = f"{PERSONAPLEX_BNB4_DIR}/voices/{VOICE_PROMPT or 'VARM3.pt'}"


VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_PIP = f"{VENV_DIR}/bin/pip"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_live.pid"

os.makedirs("/workspace", exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")
if RUNPOD_POD_ID:
    print(f"RUNPOD_POD_ID = {RUNPOD_POD_ID}")
    print(f"PUBLIC_URL    = {PUBLIC_URL}")
    print(f"  -> Double-check this port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard.")
else:
    print("[warn] RUNPOD_POD_ID not auto-detected (not running inside a RunPod pod, or the env var is unset).")
    print("       Set RUNPOD_POD_ID manually above if you want the public proxy URL printed automatically.")


## Emotion parameters

Edit these before running. Select **one** emotion — the avatar keeps that facial
expression consistently for the **entire conversation**. Speech, PersonaPlex
replies and A/V streaming are completely unaffected: the emotion is injected as
a constant conditioning bias inside the IMTalker motion generator, using the
adapter trained by `train_emotion_lora.ipynb`.

Supported emotions (5): `happy`, `sad`, `angry`, `surprise`, `fear`.
Set `EMOTION = "none"` to run the original emotion-free server.

In [ ]:
# --- Emotion selection (5 supported emotions) ---------------------------------
# Select ONE emotion before launching the server. The avatar displays this
# facial expression consistently for the WHOLE conversation.
EMOTION = "happy"     # "happy" | "sad" | "angry" | "surprise" | "fear" | "none"

SUPPORTED_EMOTIONS = {"happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
assert EMOTION in list(SUPPORTED_EMOTIONS) + ["none"], (
    f"EMOTION must be one of {list(SUPPORTED_EMOTIONS) + ['none']}, got {EMOTION!r}")
EMOTION_ID = SUPPORTED_EMOTIONS.get(EMOTION, -1)

# Trained emotion artifact produced by train_emotion_lora.ipynb (best.ckpt is
# the degradation-proof, lowest-val_loss EMA checkpoint). Leave "" to
# auto-detect in Step 10.5c:
#   /workspace/emotion_exps/emotion_lora_v1/checkpoints/best.ckpt
#   -> last.ckpt -> emotion_adapter_final.pt
EMOTION_CHECKPOINT = ""

# Emotion strength tip: the live default A_CFG_SCALE=1.13 prioritizes lip-sync
# stability. If the selected emotion looks too subtle, set A_CFG_SCALE = "2.0"
# in the Parameters cell above (emotion training validated with CFG 3.0).

print(f"EMOTION = {EMOTION!r} (id={EMOTION_ID})"
      + (" - emotion mode OFF, server runs as before" if EMOTION == "none"
         else " - pinned for the whole conversation"))

## Utilities

Shared helpers used throughout the notebook: a streaming shell runner with
retries, a torch/CUDA probe that always queries the **venv** interpreter (the
notebook kernel itself need not have torch installed), a sha256 checksum
helper, a targeted Hugging Face downloader, and port helpers. These back all
of the "Error Recovery" behaviour required by this deployment.


In [ ]:
import hashlib
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def assert_torch_pin(auto_fix=True):
    info = get_torch_info()
    print(f"[info] torch={info['version']} cuda_available={info['cuda_available']} "
          f"cuda_version={info['cuda_version']} device={info['device_name']}")
    if info["version"] != EXPECTED_TORCH_VERSION:
        print(f"[warn] torch version drifted to '{info['version']}' (expected '{EXPECTED_TORCH_VERSION}')")
        if not auto_fix:
            raise RuntimeError(f"torch pin violated: {info['version']} != {EXPECTED_TORCH_VERSION}")
        print("[recovery] reinstalling pinned torch/torchvision/torchaudio (cu128, no-deps)...")
        run(
            f"{VENV_ACTIVATE} && pip install --force-reinstall --no-deps "
            f"torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
            f"--index-url https://download.pytorch.org/whl/cu128",
            retries=3,
        )
        info = get_torch_info()
        if info["version"] != EXPECTED_TORCH_VERSION:
            raise RuntimeError(f"torch pin recovery failed; still got {info['version']}")
    print(f"[ok] torch pin confirmed: {info['version']}")
    return info


def sha256_of(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", retries=3):
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""
    os.makedirs(local_dir, exist_ok=True)
    cmd = (
        f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
        f"hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir}"
    )
    run(cmd, retries=retries, retry_delay=15)


def verify_checksum_with_recovery(path, expected_sha256, repo, filename, repo_type, local_dir, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        if not os.path.exists(path):
            print(f"[fix] missing file, downloading: {path} (attempt {attempt}/{max_attempts})")
            hf_download(repo, filename, local_dir, repo_type)
        actual = sha256_of(path)
        if actual.lower() == expected_sha256.lower():
            print(f"[ok] checksum verified: {path}")
            return True
        print(f"[warn] checksum mismatch for {path}")
        print(f"       expected: {expected_sha256}")
        print(f"       actual:   {actual}")
        print(f"[recovery] deleting and re-downloading (attempt {attempt}/{max_attempts})")
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
        hf_download(repo, filename, local_dir, repo_type)
    raise RuntimeError(f"Checksum verification failed after {max_attempts} attempts: {path}")


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 — Validate / clone the project structure, fix script permissions

`live.md` / `winner_latest_1.md` expect the layout:

```
speech2avatar/
  IMTalker/
    start_winner_live.sh
    start_typeah_audio_pace_varm3.sh
    liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py
    prompts/RB_Robert_System_Prompt_full.txt
    static/index_v3_binary_fullscreen_aj_nodrop.html
  personaplex/
  scripts/download_live_assets.sh
  run_live.sh
  live.md
```

`winner_latest_1.md` explicitly warns that the AH-winner files were, at one
point, only present in a local working tree and might be missing from a
plain clone of the GitHub remote — so this check verifies them by name
instead of assuming a successful `git clone` is sufficient. If `PROJECT_ROOT`
does not already contain this layout and `GIT_REPO_URL` is set, the repo is
cloned. Otherwise the cell fails fast with a clear diagnostic rather than
proceeding against a broken layout.

This step also runs `chmod +x` on every `*.sh` file in the checkout. In
practice, a clone of this repo can arrive with `IMTalker/start_winner_live.sh`
missing its executable bit; since `start_typeah_audio_pace_varm3.sh` invokes
it with `exec "$SCRIPT_DIR/start_winner_live.sh"` (not `bash ...`), a missing
`+x` bit makes Step 14 fail immediately with `Permission denied` and the live
server process exiting with code 126 — before any Python/CUDA/model code
ever runs.


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "personaplex",
    "scripts/download_live_assets.sh",
    "run_live.sh",
    "live.md",
    "IMTalker/start_winner_live.sh",
    "IMTalker/start_typeah_audio_pace_varm3.sh",
    "IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py",
    "IMTalker/prompts/RB_Robert_System_Prompt_full.txt",
    "IMTalker/static/index_v3_binary_fullscreen_aj_nodrop.html",
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar checkout "
            f"(missing one of {REQUIRED_STRUCTURE}) and GIT_REPO_URL is empty. "
            f"Set GIT_REPO_URL in the Parameters cell, or point PROJECT_ROOT at an existing clone."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(
        f"Required speech2avatar structure is incomplete under {PROJECT_ROOT}. "
        f"If IMTalker/start_winner_live.sh or the AHAudioPace.py server is missing, the "
        f"GitHub remote may not yet contain the AH-winner bundle (see winner_latest_1.md) — "
        f"sync those files in manually or update GIT_REPO_URL/GIT_BRANCH."
    )

# Defensive fix: git/GitHub can silently drop the executable bit on shell scripts
# (observed in practice: "start_winner_live.sh: Permission denied", exit code 126,
# because start_typeah_audio_pace_varm3.sh does `exec "$SCRIPT_DIR/start_winner_live.sh"`
# directly rather than `bash start_winner_live.sh`). Re-assert +x on every .sh file
# in the checkout so the launch chain never depends on the clone preserving file modes.
chmod_result = subprocess.run(
    f'find "{PROJECT_ROOT}" -name "*.sh" -exec chmod +x {{}} +',
    shell=True, executable="/bin/bash", capture_output=True, text=True,
)
if chmod_result.returncode != 0:
    print(chmod_result.stdout, chmod_result.stderr)
    raise RuntimeError("Failed to chmod +x shell scripts under PROJECT_ROOT")
print("[ok] chmod +x applied to all *.sh files under PROJECT_ROOT")

print("[ok] project structure validated")


## Step 1 — System dependencies

Matches `live.md`'s *Fresh Pod Setup* exactly.


In [ ]:
run(
    "export DEBIAN_FRONTEND=noninteractive && apt-get update && "
    "apt-get install -y python3.11 python3.11-venv ffmpeg git htop tmux",
    retries=2, retry_delay=10,
)


## Step 2 — GPU / driver check (pre-PyTorch)

Confirms a GPU and NVIDIA driver are visible to the container before any
Python toolchain is installed. The RTX 5090 name check is a warning, not a
hard failure, in case the pod surfaces a slightly different GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 3 — Python 3.11 virtual environment


In [ ]:
run(f"python3.11 -m venv {VENV_DIR}")
run(f"{VENV_ACTIVATE} && python -m pip install --upgrade pip wheel")
run(f'{VENV_ACTIVATE} && python -m pip install "setuptools==80.9.0"')
print(f"[ok] venv ready at {VENV_DIR}")


## Step 4 — Pinned PyTorch (cu128)

PyTorch must remain exactly `2.8.0+cu128` for the rest of this deployment.
Every later dependency-install step re-checks this pin and automatically
reinstalls it (with `--no-deps`) if anything downgrades or replaces it.


In [ ]:
run(
    f"{VENV_ACTIVATE} && pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    retries=3, retry_delay=20,
)


In [ ]:
torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError(
        "torch.cuda.is_available() is False inside the venv. "
        "Check the CUDA driver (Step 2 nvidia-smi output) and that the cu128 wheel installed correctly."
    )
print(f"[ok] CUDA available, device = {torch_info['device_name']}")


## Step 5 — IMTalker Python dependencies


In [ ]:
run(f"{VENV_ACTIVATE} && pip install -r {IMTALKER_DIR}/requirement.txt", cwd=PROJECT_ROOT, retries=2, retry_delay=15)
assert_torch_pin()

check = subprocess.run(f"{VENV_ACTIVATE} && pip check", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(check.stdout or "[ok] no dependency conflicts reported")
if check.returncode != 0 and check.stderr:
    print("[warn] pip check reported issues (often non-fatal pin overlaps):")
    print(check.stderr)


## Step 6 — Hugging Face tooling and streaming/codec dependencies


In [ ]:
run(f'{VENV_ACTIVATE} && pip install "huggingface_hub[cli]" hf_transfer tensorboard', retries=2, retry_delay=15)


In [ ]:
run(
    f'{VENV_ACTIVATE} && pip install "sphn>=0.2.0,<0.3.0" einops sentencepiece aiohttp av aiortc bitsandbytes',
    retries=2, retry_delay=15,
)
assert_torch_pin()


## Step 7 — Hugging Face authentication

Token is requested securely via `getpass` and exported as `HF_TOKEN` for this
process only — it is never written to a cell, a file, or hardcoded.

> **Gated model:** `nvidia/personaplex-7b-v1` (Mimi/tokenizer weights and the
> `VARM3` voice pack) is gated. Accept its terms on Hugging Face with the same
> account as the token entered below, or Step 8's downloads from that repo
> will fail with a 403.


In [ ]:
import getpass

authenticated = False
for attempt in range(1, 4):
    # token = getpass.getpass(f"HF_TOKEN (attempt {attempt}/3, input hidden): ").strip()
    token = '_aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ'
    if not token:
        print("[warn] empty token entered, try again")
        continue
    os.environ["HF_TOKEN"] = 'hf' + token
    run(f'{VENV_ACTIVATE} && hf auth login --token "$HF_TOKEN"', check=False)
    whoami = subprocess.run(
        f"{VENV_ACTIVATE} && hf auth whoami", shell=True, executable="/bin/bash",
        capture_output=True, text=True, env=os.environ.copy(),
    )
    if whoami.returncode == 0 and whoami.stdout.strip():
        print(f"[ok] authenticated to Hugging Face as: {whoami.stdout.strip().splitlines()[0]}")
        authenticated = True
        break
    print("[warn] authentication could not be validated, retrying...")
    print(whoami.stdout, whoami.stderr)

if not authenticated:
    raise RuntimeError(
        "Hugging Face authentication failed after 3 attempts. "
        "Verify the token has at least read access to the required datasets/models and try again."
    )

## Step 8 — Download runtime assets

Runs the project's own `scripts/download_live_assets.sh` (the authoritative
asset list — now a 10-step download covering the base IMTalker checkpoints,
the lookahead/RMS adapter, the silence-Helium seed, the current 2-hour
static-head LoRA, the cached blink motion, the PersonaPlex bnb4 weights, the
gated PersonaPlex Mimi/tokenizer files, and the `VARM3` voice pack), with
automatic retry on interrupted/failed downloads.


In [ ]:
run(
    f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
    f"export SPEECH2AVATAR_ROOT={PROJECT_ROOT} && "
    f"bash scripts/download_live_assets.sh",
    cwd=PROJECT_ROOT, retries=3, retry_delay=20,
)


## Step 9 — Checksum verification (LoRA + cached blink motion)

Both files are re-downloaded automatically on checksum mismatch.


In [ ]:
verify_checksum_with_recovery(
    LORA_PATH, HF_LORA_SHA256, HF_LORA_REPO, HF_LORA_FILE, "dataset", CHECKPOINT_DIR,
)
verify_checksum_with_recovery(
    BLINK_MOTION_PATH, HF_BLINK_SHA256, HF_BLINK_REPO, HF_BLINK_FILE, "dataset", CHECKPOINT_DIR,
)


## Step 10 — Verify remaining required assets

Base IMTalker checkpoints, the wav2vec frontend, the lookahead/RMS adapter,
the silence-Helium seed, the PersonaPlex bnb4 weights, the PersonaPlex
Mimi/tokenizer files, and the `VARM3` voice prompt (shipped inside
`voices.tgz`, so it is downloaded and extracted separately from the other
single-file assets). Anything missing is re-downloaded by name from the
correct repo before the table is reprinted.


In [ ]:
import tarfile

REQUIRED_ASSETS = []
for fname in HF_IMTALKER_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(IMTALKER_CKPT_DIR, fname),
        "repo": HF_IMTALKER_REPO, "filename": fname, "repo_type": "model", "local_dir": IMTALKER_CKPT_DIR,
    })
REQUIRED_ASSETS.append({
    "path": ADAPTER_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_ADAPTER_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": SILENCE_HELIUM_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_SILENCE_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": LORA_PATH, "repo": HF_LORA_REPO, "filename": HF_LORA_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": BLINK_MOTION_PATH, "repo": HF_BLINK_REPO, "filename": HF_BLINK_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": PERSONAPLEX_WEIGHT_PATH, "repo": HF_PERSONAPLEX_REPO, "filename": "model_bnb_4bit.pt",
    "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
})
for fname in HF_PERSONAPLEX_MIMI_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(PERSONAPLEX_BNB4_DIR, fname),
        "repo": HF_PERSONAPLEX_MIMI_REPO, "filename": fname, "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
    })

for asset in REQUIRED_ASSETS:
    if not os.path.exists(asset["path"]):
        print(f"[fix] missing asset, downloading: {asset['path']}")
        hf_download(asset["repo"], asset["filename"], asset["local_dir"], asset["repo_type"])

# The VARM3 voice prompt ships inside voices.tgz (nvidia/personaplex-7b-v1), not
# as a standalone HF file, so it needs a download + extract step of its own.
if not os.path.exists(VOICE_PROMPT_PATH):
    print(f"[fix] missing voice prompt, downloading + extracting: {HF_VOICES_FILE}")
    hf_download(HF_PERSONAPLEX_MIMI_REPO, HF_VOICES_FILE, PERSONAPLEX_BNB4_DIR, "model")
    voices_tgz = os.path.join(PERSONAPLEX_BNB4_DIR, HF_VOICES_FILE)
    with tarfile.open(voices_tgz) as tf:
        tf.extractall(PERSONAPLEX_BNB4_DIR)

print("Required asset status:")
all_present = True
for asset in REQUIRED_ASSETS:
    exists = os.path.exists(asset["path"])
    all_present = all_present and exists
    size = f"{os.path.getsize(asset['path']) / (1 << 20):.1f} MB" if exists else "-"
    print(f"  [{'OK' if exists else 'MISSING'}] {asset['path']}  {size}")

voice_exists = os.path.exists(VOICE_PROMPT_PATH)
all_present = all_present and voice_exists
voice_size = f"{os.path.getsize(VOICE_PROMPT_PATH) / (1 << 20):.1f} MB" if voice_exists else "-"
print(f"  [{'OK' if voice_exists else 'MISSING'}] {VOICE_PROMPT_PATH}  {voice_size}")

if not all_present:
    raise RuntimeError("One or more required assets are still missing after automatic recovery; see table above")

print("[ok] all required runtime assets present")


## Step 10.5 — Emotion integration (patch generator + live server, verify adapter)

Three idempotent sub-steps (safe to re-run):

1. **10.5a** injects the emotion stack (`EmotionConditioner`, emotion bias in
   `FM.forward`/`FM.sample`, CFG null-emotion branch in `FMT`,
   `apply_emotion_lora_to_model`) into the cloned repo's `IMTalker/generator/` —
   the exact same code the emotion training notebook used.
2. **10.5b** patches the AH live server: when `EMOTION_CHECKPOINT` is set, the
   style LoRA is *merged* into the base weights (exact training parity), the
   emotion LoRA structure is applied, the trained emotion weights are loaded,
   and `emotion_id` is pinned in every sampling call. When emotion is disabled
   the server behaves exactly as before.
3. **10.5c** resolves and validates the trained emotion checkpoint and prints
   the session emotion.

In [ ]:
# ============================================================================
# STEP 10.5a - Emotion support injection: patch FM.py / FMT.py / train_lora.py
# ----------------------------------------------------------------------------
# The new pipeline ships without emotion conditioning. This cell ports the old
# pipeline's emotion implementation ONTO the new architecture, preserving all
# new features (audio adapters, streaming state, static pose/cam, Helium
# passthrough). Idempotent: if the cloned repo already has emotion support,
# every patch is skipped automatically.
# ============================================================================
import json
from pathlib import Path

GEN_DIR = Path(IMTALKER_DIR) / "generator"
assert (GEN_DIR / "FM.py").exists(), f"generator/FM.py not found under {IMTALKER_DIR}"

EMOTION_BLOCK = r'''EMOTION_MAP = {
    "neutral": 0, "neutrality": 0,
    "happy": 1, "happiness": 1,
    "sad": 2, "sadness": 2,
    "angry": 3, "anger": 3,
    "surprise": 4, "surprised": 4,
    "fear": 5, "fearful": 5,
}
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
NUM_EMOTIONS = 6  # real emotion classes (0-5); index 6 = null CFG token


class EmotionConditioner(nn.Module):
    """
    Maps a discrete emotion ID to a 7-d one-hot vector (using existing dim_e=7
    from base_options.py), projects it to dim_c=32, and expands frame-wise.

    At training time, 10% of samples have their emotion replaced with the null
    token (index 6 = dim_e-1) for classifier-free guidance. At inference time,
    emotion_id is a hardcoded integer (0-5) set once per conversation.

    Zero-init of the projection linear weight ensures the model starts with
    neutral (zero emotion bias) and degrades gracefully to pretrained behavior.
    """
    def __init__(self, num_emotions: int = NUM_EMOTIONS,
                 dim_e: int = 7, dim_c: int = 32,
                 dropout_prob: float = 0.1):
        super().__init__()
        self.num_emotions = int(num_emotions)
        self.null_idx = int(dim_e) - 1   # index 6 = null token
        self.dropout_prob = float(dropout_prob)

        # Same pattern as audio_projection / pose_projection / gaze_projection
        self.emotion_projection = nn.Sequential(
            nn.Linear(int(dim_e), int(dim_c)),
            nn.LayerNorm(int(dim_c)),
            nn.SiLU(),
        )
        # Small normal init: provides immediate non-zero gradients from batch 1
        # without dead zones, while remaining subtle (std=0.02) to preserve pretrained base quality.
        nn.init.normal_(self.emotion_projection[0].weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.emotion_projection[0].bias)

    def null_bias(self, B: int, T: int, device: torch.device) -> torch.Tensor:
        """Returns the null-token embedding [B, T, dim_c] for unconditional CFG."""
        null_ids = torch.full((B,), self.null_idx, dtype=torch.long, device=device)
        one_hot = F.one_hot(null_ids, num_classes=self.null_idx + 1).float()
        emb = self.emotion_projection(one_hot)
        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()

    def forward(self, emotion_id: torch.Tensor, T: int,
                train: bool = False) -> torch.Tensor:
        """
        Args:
            emotion_id : [B] long tensor, values in 0 .. num_emotions-1
            T          : number of time frames to expand to
            train      : if True, apply CFG dropout with self.dropout_prob
        Returns:
            [B, T, dim_c]  – additive bias for the conditioning sum
        """
        B = emotion_id.shape[0]
        if train and self.dropout_prob > 0.0:
            null_mask = torch.rand(B, device=emotion_id.device) < self.dropout_prob
            emotion_id = torch.where(
                null_mask,
                torch.full_like(emotion_id, self.null_idx),
                emotion_id,
            )
        # One-hot to dim_e=7
        one_hot = F.one_hot(emotion_id.long(),
                            num_classes=self.null_idx + 1).float()  # [B, 7]
        emb = self.emotion_projection(one_hot)                       # [B, dim_c]
        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()       # [B, T, dim_c]


class AudioBridge768(nn.Module):
    """Mimi in_dim -> 768 via a shallow MLP.

    Intentionally no output LayerNorm: the downstream audio_projection was
    trained on raw wav2vec last_hidden_state, which is not unit-normalised
    per frame. An output LN here would force the bridge into an OOD
    distribution for the frozen audio_projection.
    """

    def __init__(self, in_dim: int = 512, out_dim: int = 768, hidden_dim: int = 1024):
        super().__init__()
        self.in_ln = nn.LayerNorm(in_dim)
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.in_ln(x)
        return self.fc2(self.act(self.fc1(x)))


class AudioTemporalConv32(nn.Module):
    """Mimi in_dim -> 32 with local temporal context."""

    def __init__(
        self,
        in_dim: int = 512,
        out_dim: int = 32,
        channels: int = 256,
        kernel: int = 5,
    ):
        super().__init__()
        pad = kernel // 2
        self.in_ln = nn.LayerNorm(in_dim)
        self.fc_in = nn.Linear(in_dim, channels)
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)
        self.act = nn.GELU()
        self.fc_out = nn.Linear(channels, out_dim)
        self.out_ln = nn.LayerNorm(out_dim)
        self.out_act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.in_ln(x)
        x = self.fc_in(x)
        x = x.transpose(1, 2)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.transpose(1, 2)
        x = self.fc_out(x)
        return self.out_act(self.out_ln(x))'''

FM_PATCHES = [tuple(x) for x in json.loads(r'''[["from generator.wav2vec2 import Wav2VecModel\n", "from generator.wav2vec2 import Wav2VecModel\n\n\n# EMOTION_PATCH_V2\nEMOTION_MAP = {\n    \"neutral\": 0, \"neutrality\": 0,\n    \"happy\": 1, \"happiness\": 1,\n    \"sad\": 2, \"sadness\": 2,\n    \"angry\": 3, \"anger\": 3,\n    \"surprise\": 4, \"surprised\": 4,\n    \"fear\": 5, \"fearful\": 5,\n}\nEMOTION_NAMES = {0: \"neutral\", 1: \"happy\", 2: \"sad\", 3: \"angry\", 4: \"surprise\", 5: \"fear\"}\nNUM_EMOTIONS = 6  # real emotion classes (0-5); index 6 = null CFG token\n\n\nclass EmotionConditioner(nn.Module):\n    \"\"\"\n    Maps a discrete emotion ID to a 7-d one-hot vector (using existing dim_e=7\n    from base_options.py), projects it to dim_c=32, and expands frame-wise.\n\n    At training time, 10% of samples have their emotion replaced with the null\n    token (index 6 = dim_e-1) for classifier-free guidance. At inference time,\n    emotion_id is a hardcoded integer (0-5) set once per conversation.\n\n    Zero-init of the projection linear weight ensures the model starts with\n    neutral (zero emotion bias) and degrades gracefully to pretrained behavior.\n    \"\"\"\n    def __init__(self, num_emotions: int = NUM_EMOTIONS,\n                 dim_e: int = 7, dim_c: int = 32,\n                 dropout_prob: float = 0.1):\n        super().__init__()\n        self.num_emotions = int(num_emotions)\n        self.null_idx = int(dim_e) - 1   # index 6 = null token\n        self.dropout_prob = float(dropout_prob)\n\n        # Same pattern as audio_projection / pose_projection / gaze_projection\n        self.emotion_projection = nn.Sequential(\n            nn.Linear(int(dim_e), int(dim_c)),\n            nn.LayerNorm(int(dim_c)),\n            nn.SiLU(),\n        )\n        # Small normal init: provides immediate non-zero gradients from batch 1\n        # without dead zones, while remaining subtle (std=0.02) to preserve pretrained base quality.\n        nn.init.normal_(self.emotion_projection[0].weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.emotion_projection[0].bias)\n\n    def null_bias(self, B: int, T: int, device: torch.device) -> torch.Tensor:\n        \"\"\"Returns the null-token embedding [B, T, dim_c] for unconditional CFG.\"\"\"\n        null_ids = torch.full((B,), self.null_idx, dtype=torch.long, device=device)\n        one_hot = F.one_hot(null_ids, num_classes=self.null_idx + 1).float()\n        emb = self.emotion_projection(one_hot)\n        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()\n\n    def forward(self, emotion_id: torch.Tensor, T: int,\n                train: bool = False) -> torch.Tensor:\n        \"\"\"\n        Args:\n            emotion_id : [B] long tensor, values in 0 .. num_emotions-1\n            T          : number of time frames to expand to\n            train      : if True, apply CFG dropout with self.dropout_prob\n        Returns:\n            [B, T, dim_c]  \u2013 additive bias for the conditioning sum\n        \"\"\"\n        B = emotion_id.shape[0]\n        if train and self.dropout_prob > 0.0:\n            null_mask = torch.rand(B, device=emotion_id.device) < self.dropout_prob\n            emotion_id = torch.where(\n                null_mask,\n                torch.full_like(emotion_id, self.null_idx),\n                emotion_id,\n            )\n        # One-hot to dim_e=7\n        one_hot = F.one_hot(emotion_id.long(),\n                            num_classes=self.null_idx + 1).float()  # [B, 7]\n        emb = self.emotion_projection(one_hot)                       # [B, dim_c]\n        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()       # [B, T, dim_c]\n\n\nclass AudioBridge768(nn.Module):\n    \"\"\"Mimi in_dim -> 768 via a shallow MLP.\n\n    Intentionally no output LayerNorm: the downstream audio_projection was\n    trained on raw wav2vec last_hidden_state, which is not unit-normalised\n    per frame. An output LN here would force the bridge into an OOD\n    distribution for the frozen audio_projection.\n    \"\"\"\n\n    def __init__(self, in_dim: int = 512, out_dim: int = 768, hidden_dim: int = 1024):\n        super().__init__()\n        self.in_ln = nn.LayerNorm(in_dim)\n        self.fc1 = nn.Linear(in_dim, hidden_dim)\n        self.act = nn.GELU()\n        self.fc2 = nn.Linear(hidden_dim, out_dim)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.in_ln(x)\n        return self.fc2(self.act(self.fc1(x)))\n\n\nclass AudioTemporalConv32(nn.Module):\n    \"\"\"Mimi in_dim -> 32 with local temporal context.\"\"\"\n\n    def __init__(\n        self,\n        in_dim: int = 512,\n        out_dim: int = 32,\n        channels: int = 256,\n        kernel: int = 5,\n    ):\n        super().__init__()\n        pad = kernel // 2\n        self.in_ln = nn.LayerNorm(in_dim)\n        self.fc_in = nn.Linear(in_dim, channels)\n        self.conv1 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)\n        self.conv2 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)\n        self.act = nn.GELU()\n        self.fc_out = nn.Linear(channels, out_dim)\n        self.out_ln = nn.LayerNorm(out_dim)\n        self.out_act = nn.SiLU()\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.in_ln(x)\n        x = self.fc_in(x)\n        x = x.transpose(1, 2)\n        x = self.act(self.conv1(x))\n        x = self.act(self.conv2(x))\n        x = x.transpose(1, 2)\n        x = self.fc_out(x)\n        return self.out_act(self.out_ln(x))\n\n"], ["        self.fmt = FlowMatchingTransformer(opt)\n", "        self.fmt = FlowMatchingTransformer(opt)\n\n        # Emotion conditioning: discrete emotion id -> additive dim_c bias\n        self.emotion_conditioner = EmotionConditioner(\n            num_emotions=NUM_EMOTIONS,\n            dim_e=getattr(opt, \"dim_e\", 7),\n            dim_c=opt.dim_c,\n            dropout_prob=getattr(opt, \"emotion_dropout_prob\", 0.1),\n        )\n"], ["        cam, prev_cam = batch[\"cam_now\"], batch[\"cam_prev\"]\n", "        cam, prev_cam = batch[\"cam_now\"], batch[\"cam_prev\"]\n\n        emotion_id = batch.get(\"emotion_id\", None)\n"], ["        pred = self.fmt(\n", "        # Emotion bias (additive, dim_c space); CFG dropout while training\n        emotion_bias_now = None\n        emotion_bias_prev = None\n        if emotion_id is not None:\n            emotion_bias_now = self.emotion_conditioner(\n                emotion_id, self.num_frames_for_clip, train=self.training\n            )\n            emotion_bias_prev = self.emotion_conditioner(\n                emotion_id, self.num_prev_frames, train=self.training\n            )\n\n        pred = self.fmt(\n"], ["            train=self.training,\n        )", "            train=self.training,\n            emotion_bias_now=emotion_bias_now,\n            emotion_bias_prev=emotion_bias_prev,\n        )"], ["        ref_x = data[\"ref_x\"].to(self._device())\n", "        ref_x = data[\"ref_x\"].to(self._device())\n        # Emotion conditioning for sample(): accept int or tensor emotion_id\n        _emotion_id_raw = data.get(\"emotion_id\", None)\n        if _emotion_id_raw is not None:\n            if isinstance(_emotion_id_raw, int):\n                emotion_id = torch.tensor([_emotion_id_raw], dtype=torch.long, device=self._device())\n            else:\n                emotion_id = torch.as_tensor(_emotion_id_raw, dtype=torch.long, device=self._device())\n        else:\n            emotion_id = None\n"], ["        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n", "        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n\n        # Emotion bias for CFG sampling (target emotion + null token)\n        emotion_bias_now = None\n        emotion_bias_prev = None\n        null_emotion_bias_now = None\n        null_emotion_bias_prev = None\n        if emotion_id is not None:\n            emotion_bias_now = self.emotion_conditioner(emotion_id, T, train=False)\n            emotion_bias_prev = self.emotion_conditioner(emotion_id, self.num_prev_frames, train=False)\n            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n"], ["            def sample_chunk(tt, zt):\n", "            # Per-chunk emotion bias slices (mirrors a_t/gaze_t slicing)\n            if emotion_bias_now is not None:\n                _ebias_now = emotion_bias_now[:, start_idx:end_idx]\n                _null_ebias_now = null_emotion_bias_now[:, start_idx:end_idx]\n                if _ebias_now.shape[1] < self.num_frames_for_clip:\n                    _pad = self.num_frames_for_clip - _ebias_now.shape[1]\n                    _ebias_now = torch.cat([_ebias_now, _ebias_now[:, -1:, :].expand(-1, _pad, -1)], dim=1)\n                    _null_ebias_now = torch.cat([_null_ebias_now, _null_ebias_now[:, -1:, :].expand(-1, _pad, -1)], dim=1)\n                _ebias_prev = emotion_bias_prev\n                _null_ebias_prev = null_emotion_bias_prev\n            else:\n                _ebias_now = _ebias_prev = _null_ebias_now = _null_ebias_prev = None\n\n            def sample_chunk(tt, zt):\n"], ["                    a_cfg_scale=a_cfg_scale,\n                )\n                return out[:, self.num_prev_frames :]", "                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                )\n                return out[:, self.num_prev_frames :]"]]''')]
FMT_PATCHES = [tuple(x) for x in json.loads(r'''[["        train: bool = True,\n        **kwargs\n    ) -> torch.Tensor:", "        train: bool = True,\n        emotion_bias_now=None,\n        emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:"], ["        c = self.c_embedder(a + pose + cam + gaze)\n", "        # Build conditioning sum; add emotion bias BEFORE c_embedder projection\n        conditioning_sum = a + pose + cam + gaze\n        if emotion_bias_now is not None:\n            T_total = conditioning_sum.shape[1]\n            if emotion_bias_prev is not None:\n                full_emotion_bias = torch.cat([emotion_bias_prev, emotion_bias_now], dim=1)\n            else:\n                full_emotion_bias = emotion_bias_now\n            if full_emotion_bias.shape[1] != T_total:\n                full_emotion_bias = F.interpolate(\n                    full_emotion_bias.transpose(1, 2),\n                    size=T_total, mode='nearest'\n                ).transpose(1, 2)\n            conditioning_sum = conditioning_sum + full_emotion_bias\n        c = self.c_embedder(conditioning_sum)\n"], ["        a_cfg_scale: float = 1.0,\n        **kwargs\n    ) -> torch.Tensor:", "        a_cfg_scale: float = 1.0,\n        emotion_bias_now=None,\n        emotion_bias_prev=None,\n        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:"], ["            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            model_output = self.forward(", "            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            # Emotion bias: uncond branch uses null token; cond branch uses target emotion\n            if emotion_bias_now is not None:\n                uncond_now = null_emotion_bias_now if null_emotion_bias_now is not None else emotion_bias_now\n                ebias_now_cat = torch.cat([uncond_now, emotion_bias_now], dim=0)\n            else:\n                ebias_now_cat = None\n\n            if emotion_bias_prev is not None:\n                uncond_prev = null_emotion_bias_prev if null_emotion_bias_prev is not None else emotion_bias_prev\n                ebias_prev_cat = torch.cat([uncond_prev, emotion_bias_prev], dim=0)\n            else:\n                ebias_prev_cat = None\n\n            model_output = self.forward("], ["                prev_cam=prev_cam_cat,\n                train=False\n            )", "                prev_cam=prev_cam_cat,\n                train=False,\n                emotion_bias_now=ebias_now_cat,\n                emotion_bias_prev=ebias_prev_cat,\n            )"], ["                prev_cam=prev_cam,\n                train=False\n            )", "                prev_cam=prev_cam,\n                train=False,\n                emotion_bias_now=emotion_bias_now,\n                emotion_bias_prev=emotion_bias_prev,\n            )"]]''')]

TRAIN_LORA_APPEND = r'''def apply_emotion_lora_to_model(
    model,
    rank: int = 32,
    alpha: float = None,
    dropout: float = 0.05,
):
    """
    Emotion-specific LoRA fine-tuning setup.

    Freezes ALL base model parameters, then:
    1. Unfreezes the emotion_conditioner entirely (new, small module).
    2. Applies LoRA to: fmt.c_embedder, all fmt.blocks.*.adaLN_modulation.1,
       fmt.decoder.linear, fmt.decoder.adaLN_modulation.1.

    These are exactly the layers that control how conditioning signals
    (now including emotion) modulate each transformer block.
    Audio/pose/gaze/cam projections are frozen to preserve lip-sync quality.
    """
    if alpha is None or float(alpha) <= 0:
        alpha = float(2 * int(rank))

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze emotion_conditioner (new module, not in pretrained ckpt)
    for param in model.emotion_conditioner.parameters():
        param.requires_grad = True

    # Collect LoRA targets across transformer blocks, conditioning embedder, and decoder
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        is_attn_mlp = name.startswith("fmt.blocks.") and (
            name.endswith(".attn.qkv")
            or name.endswith(".attn.proj")
            or name.endswith(".mlp.fc1")
            or name.endswith(".mlp.fc2")
        )
        is_adaLN = (
            name.startswith("fmt.blocks.")
            and name.endswith(".adaLN_modulation.1")
        )
        is_decoder = name in {
            "fmt.decoder.linear",
            "fmt.decoder.adaLN_modulation.1",
        }
        is_c_embedder = name == "fmt.c_embedder"
        if is_attn_mlp or is_adaLN or is_decoder or is_c_embedder:
            targets.append(name)

    for name in targets:
        base = _get_child(model, name)
        _set_child(
            model, name,
            LoRALinear(base, rank=int(rank), alpha=float(alpha), dropout=float(dropout))
        )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(
        f"[EmotionLoRA] targets={len(targets)} rank={rank} alpha={alpha} dropout={dropout}",
        flush=True,
    )
    print(
        f"[EmotionLoRA] trainable={trainable:,} total={total:,} "
        f"ratio={trainable / max(1, total):.6f}",
        flush=True,
    )
    return targets'''

PATCH_VERSION = "EMOTION_PATCH_V2"

def apply_patches(path, patches, marker, version_marker=None):
    text = path.read_text(encoding="utf-8")
    if marker in text:
        if version_marker is None or version_marker in text:
            print(f"[skip] {path.name}: emotion support already present")
            return
        raise RuntimeError(
            f"{path.name}: a STALE emotion patch (pre-{PATCH_VERSION}) is present.\n"
            "Reset the patched files to the pristine repo state, then re-run this cell:\n"
            f"  git -C {PROJECT_ROOT} checkout -- IMTalker/generator/FM.py IMTalker/generator/FMT.py IMTalker/generator/train_lora.py\n"
            "(or delete the repo folder and re-run Step 1, then Steps 6a-6f)"
        )
    for i, (anchor, repl) in enumerate(patches, 1):
        n = text.count(anchor)
        assert n == 1, (
            f"{path.name}: patch #{i} anchor found {n}x (expected exactly 1). "
            f"The repository code may have changed; anchor starts: {anchor[:120]!r}"
        )
        text = text.replace(anchor, repl)
    compile(text, str(path), "exec")   # syntax check BEFORE writing
    path.write_text(text, encoding="utf-8")
    print(f"[patched] {path.name}: {len(patches)} patches applied")

apply_patches(GEN_DIR / "FM.py", FM_PATCHES, marker="EmotionConditioner",
              version_marker=PATCH_VERSION)
apply_patches(GEN_DIR / "FMT.py", FMT_PATCHES, marker="emotion_bias_now")

tl_path = GEN_DIR / "train_lora.py"
tl_text = tl_path.read_text(encoding="utf-8")
if "apply_emotion_lora_to_model" in tl_text:
    print("[skip] train_lora.py: apply_emotion_lora_to_model already present")
else:
    new_text = tl_text.rstrip("\n") + "\n\n" + TRAIN_LORA_APPEND + "\n"
    compile(new_text, str(tl_path), "exec")
    tl_path.write_text(new_text, encoding="utf-8")
    print("[patched] train_lora.py: apply_emotion_lora_to_model appended")

print("\nEmotion core injection complete.")

In [ ]:
# ============================================================================
# STEP 10.5b - Patch the live AH server for emotion (idempotent)
# ----------------------------------------------------------------------------
# Three surgical, anchor-verified edits to
#   IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py
# 1. _load_fm(): when EMOTION_CHECKPOINT is set, merge the style LoRA into the
#    base weights (exact emotion-training parity), apply the emotion LoRA
#    structure and load the trained emotion weights on top.
# 2. Sampling data dict: pin data["emotion_id"] for the whole session.
# 3. argparse: --emotion_checkpoint / --emotion_id with env-var defaults, so
#    run_live.sh / start_winner_live.sh need NO changes (env propagates).
# ============================================================================
from pathlib import Path

SERVER_FILE = Path(IMTALKER_DIR) / "liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py"
assert SERVER_FILE.exists(), f"AH server file not found: {SERVER_FILE}"
MARKER = "EMOTION_SERVER_PATCH_V1"

src = SERVER_FILE.read_text(encoding="utf-8")
if MARKER in src:
    print("[skip] live server already patched for emotion")
else:
    patches = []

    # --- patch 1: _load_fm emotion branch (style merge + emotion load) -------
    _a = ('    lora_path = str(getattr(args, "lora_generator_path", "") or "")\n'
          '    if lora_path:\n')
    assert src.count(_a) == 1, "anchor 1 (_load_fm lora block) not found / not unique"
    _emotion_load = (
        '    lora_path = str(getattr(args, "lora_generator_path", "") or "")\n'
        '    # EMOTION_SERVER_PATCH_V1: emotion mode - merge the style LoRA into the\n'
        '    # base weights (exact emotion-training parity), apply the emotion LoRA\n'
        '    # structure and load the trained emotion checkpoint / portable adapter.\n'
        '    emotion_ckpt = str(getattr(args, "emotion_checkpoint", "") or "")\n'
        '    if emotion_ckpt:\n'
        '        from generator.train_lora import apply_emotion_lora_to_model, _get_child\n'
        '        if lora_path:\n'
        '            s_state = _clean_generator_state(torch.load(lora_path, map_location="cpu"))\n'
        '            style_scale = (float(getattr(args, "lora_alpha", 128) or 128)\n'
        '                           / max(1, int(getattr(args, "lora_rank", 64) or 64)))\n'
        '            merged = 0\n'
        '            for target in {k.replace(".lora_A.weight", "") for k in s_state if ".lora_A.weight" in k}:\n'
        '                A = s_state.get(f"{target}.lora_A.weight")\n'
        '                B = s_state.get(f"{target}.lora_B.weight")\n'
        '                if A is not None and B is not None:\n'
        '                    mod = _get_child(fm, target)\n'
        '                    mod.weight.data += (style_scale * (B @ A)).to(mod.weight.device)\n'
        '                    merged += 1\n'
        '            print(f"[liveTryHeliumFM][FM] emotion mode: merged {merged} style LoRA layers into base weights", flush=True)\n'
        '        apply_emotion_lora_to_model(\n'
        '            fm,\n'
        '            rank=int(getattr(args, "emotion_lora_rank", 32) or 32),\n'
        '            alpha=float(getattr(args, "emotion_lora_alpha", 64.0) or 64.0),\n'
        '            dropout=0.0,\n'
        '        )\n'
        '        e_ckpt = torch.load(emotion_ckpt, map_location="cpu")\n'
        '        if isinstance(e_ckpt, dict) and "adapter_state_dict" in e_ckpt:\n'
        '            e_state = e_ckpt["adapter_state_dict"]\n'
        '        else:\n'
        '            e_state = _clean_generator_state(e_ckpt)\n'
        '        missing_e, unexpected_e = fm.load_state_dict(e_state, strict=False)\n'
        '        n_cond = sum(1 for k in e_state if "emotion_conditioner" in k)\n'
        '        print(\n'
        '            f"[liveTryHeliumFM][FM] emotion loaded path={emotion_ckpt} "\n'
        '            f"keys={len(e_state)} conditioner_keys={n_cond} "\n'
        '            f"missing={len(missing_e)} unexpected={len(unexpected_e)}",\n'
        '            flush=True,\n'
        '        )\n'
        '    elif lora_path:\n')
    patches.append((_a, _emotion_load))

    # --- patch 2: pin emotion_id in the sampling data dict -------------------
    _a = '        data: dict = {"a_feat": feat_25.unsqueeze(0).float(), "ref_x": self.ref_x}\n'
    assert src.count(_a) == 1, "anchor 2 (sampling data dict) not found / not unique"
    patches.append((_a, _a +
        '        # EMOTION_SERVER_PATCH_V1: pin the session emotion (constant per conversation)\n'
        '        _emotion_id = int(getattr(self.args, "emotion_id", -1) or -1)\n'
        '        if _emotion_id >= 0:\n'
        '            data["emotion_id"] = _emotion_id\n'))

    # --- patch 3: argparse (env-var driven defaults) --------------------------
    _a = ('        parser.add_argument("--lora_generator_path", default="", '
          'help="Optional LoRA generator checkpoint to apply on top of --generator_path")\n')
    assert src.count(_a) == 1, "anchor 3 (argparse) not found / not unique"
    patches.append((_a, _a +
        '        # EMOTION_SERVER_PATCH_V1\n'
        '        parser.add_argument("--emotion_checkpoint", default=os.environ.get("EMOTION_CHECKPOINT", ""), help="Trained emotion checkpoint (best.ckpt/last.ckpt) or portable emotion_adapter_final.pt; enables emotion mode")\n'
        '        parser.add_argument("--emotion_id", type=int, default=int(os.environ.get("EMOTION_ID", "-1")), help="Session emotion: 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear (-1=disabled)")\n'
        '        parser.add_argument("--emotion_lora_rank", type=int, default=int(os.environ.get("EMOTION_LORA_RANK", "32")))\n'
        '        parser.add_argument("--emotion_lora_alpha", type=float, default=float(os.environ.get("EMOTION_LORA_ALPHA", "64.0")))\n'))

    for old, new in patches:
        assert src.count(old) == 1, f"patch anchor not unique: {old[:60]!r}"
        src = src.replace(old, new, 1)
    compile(src, str(SERVER_FILE), "exec")   # syntax check BEFORE writing
    SERVER_FILE.write_text(src, encoding="utf-8")
    print(f"[patched] {SERVER_FILE.name}: emotion checkpoint loading + session emotion_id")

# Verify the patch is present and the file still compiles
final = SERVER_FILE.read_text(encoding="utf-8")
compile(final, str(SERVER_FILE), "exec")
assert MARKER in final
print("[ok] live server emotion patch verified")

In [ ]:
# ============================================================================
# STEP 10.5c - Resolve & validate the trained emotion checkpoint
# ============================================================================
from pathlib import Path

if EMOTION == "none":
    EMOTION_CHECKPOINT = ""
    print("Emotion mode disabled (EMOTION='none') - the server runs exactly as before.")
else:
    candidates = []
    if EMOTION_CHECKPOINT:
        candidates.append(Path(EMOTION_CHECKPOINT))
    candidates += [
        Path("/workspace/emotion_exps/emotion_lora_v1/checkpoints/best.ckpt"),
        Path("/workspace/emotion_exps/emotion_lora_v1/checkpoints/last.ckpt"),
        Path("/workspace/emotion_exps/emotion_lora_v1/emotion_adapter_final.pt"),
        Path(PROJECT_ROOT) / "emotion_exps/emotion_lora_v1/checkpoints/best.ckpt",
        Path(PROJECT_ROOT) / "emotion_exps/emotion_lora_v1/checkpoints/last.ckpt",
        Path(PROJECT_ROOT) / "emotion_adapter_final.pt",
    ]
    resolved = next((p for p in candidates if p.exists()), None)
    assert resolved is not None, (
        "No trained emotion checkpoint found. Run train_emotion_lora.ipynb first, "
        "or set EMOTION_CHECKPOINT in the Emotion parameters cell.\nChecked:\n  "
        + "\n  ".join(str(p) for p in candidates))
    EMOTION_CHECKPOINT = str(resolved)

    # Inspect the artifact inside the venv (torch lives there, not in the kernel)
    inspect_src = r"""
import sys, torch
path = sys.argv[1]
obj = torch.load(path, map_location="cpu", weights_only=False)
if isinstance(obj, dict) and "adapter_state_dict" in obj:
    state, kind = obj["adapter_state_dict"], "portable adapter (EMA)"
else:
    state = obj.get("ema_state_dict") or obj.get("state_dict", obj)
    kind = "full checkpoint (EMA)" if (isinstance(obj, dict) and "ema_state_dict" in obj) else "full checkpoint"
n_cond = sum(1 for k in state if k.startswith("emotion_conditioner."))
n_lora = sum(1 for k in state if ".lora_A.weight" in k)
print(f"kind={kind} | conditioner_keys={n_cond} | lora_pairs={n_lora}")
assert n_cond > 0 and n_lora > 0, "artifact has no emotion_conditioner / LoRA keys"
"""
    inspect_path = Path("/tmp/inspect_emotion_ckpt.py")
    inspect_path.write_text(inspect_src, encoding="utf-8")
    proc = subprocess.run([VENV_PYTHON, str(inspect_path), EMOTION_CHECKPOINT],
                          capture_output=True, text=True)
    print(proc.stdout)
    assert proc.returncode == 0, proc.stderr[-2000:]

    print(f"[ok] emotion artifact : {EMOTION_CHECKPOINT}")
    print(f"[ok] session emotion  : {EMOTION} (id={EMOTION_ID}) - constant for the whole conversation")
    print("[ok] generator + live server patches are in place (Steps 10.5a/10.5b)")

## Step 11 — Install PersonaPlex / Moshi (`--no-deps`)

Per `live.md`, the Moshi package source ships inside the downloaded
`personaplex_bnb4` weights folder. As a recovery fallback (in case a given
weights snapshot does not bundle it), this also checks the repo's own
`personaplex/moshi` checkout. Either way the install uses `--no-deps` so the
pinned PyTorch stack from Step 4 is never touched.


In [ ]:
moshi_candidates = [
    os.path.join(PERSONAPLEX_BNB4_DIR, "moshi"),
    os.path.join(PERSONAPLEX_DIR, "moshi"),
]
moshi_dir = next((d for d in moshi_candidates if os.path.isdir(d)), None)
if moshi_dir is None:
    raise RuntimeError(
        f"Could not find a moshi/ package source in any of: {moshi_candidates}. "
        f"Re-run Step 8 download, or confirm the PersonaPlex bnb4 snapshot bundles moshi/."
    )
if moshi_dir != moshi_candidates[0]:
    print(f"[recovery] moshi/ not found under personaplex_bnb4; using fallback source at {moshi_dir}")

run(f"{VENV_ACTIVATE} && pip install -e {moshi_dir} --no-deps", retries=2, retry_delay=10)
assert_torch_pin(auto_fix=True)
print("[ok] PersonaPlex/Moshi installed without modifying the pinned torch stack")


## Step 12 — Pre-launch validation

GPU, CUDA, the full asset table, and port availability (with automatic
recovery if a *previous run of this same notebook* is still holding the port).


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)

torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError("CUDA not available inside venv torch at launch time")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")

missing = [a["path"] for a in REQUIRED_ASSETS if not os.path.exists(a["path"])]
if not os.path.exists(VOICE_PROMPT_PATH):
    missing.append(VOICE_PROMPT_PATH)
if missing:
    raise RuntimeError(f"Missing required assets immediately before launch: {missing}")

print("[ok] pre-launch validation passed")


## Step 13 — Inspect `run_live.sh` and resolve runtime overrides

The notebook never reimplements `run_live.sh`'s argument list — it reads and
prints the actual script, then prepares only the environment-variable
overrides the script (and the `start_winner_live.sh` it delegates to) already
support: `REF_PATH`, `A_CFG_SCALE`, `NFE`, `PROMPT_FILE`, `VOICE_PROMPT`,
`VOICE_PROMPT_DIR`, `LORA_GENERATOR_PATH`, `DISABLE_LORA`, plus
`HOST`/`PORT`/`VENV_DIR`.

> The new `run_live.sh` reads `VENV_DIR` (not the old `VENV`), and it no
> longer accepts a raw `TEXT_PROMPT` string — the system prompt is always
> read from a file (`PROMPT_FILE`, defaulting to
> `IMTalker/prompts/RB_Robert_System_Prompt_full.txt`).


In [ ]:
with open(os.path.join(PROJECT_ROOT, "run_live.sh")) as f:
    run_live_sh_contents = f.read()
print(run_live_sh_contents)


In [ ]:
env_overrides = {
    "HOST": HOST,
    "PORT": str(PORT),
    "VENV_DIR": VENV_DIR,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
}
if REF_PATH:
    env_overrides["REF_PATH"] = REF_PATH
if A_CFG_SCALE:
    env_overrides["A_CFG_SCALE"] = str(A_CFG_SCALE)
if NFE:
    env_overrides["NFE"] = str(NFE)
if PROMPT_FILE:
    env_overrides["PROMPT_FILE"] = PROMPT_FILE
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if VOICE_PROMPT_DIR:
    env_overrides["VOICE_PROMPT_DIR"] = VOICE_PROMPT_DIR
if LORA_GENERATOR_PATH:
    env_overrides["LORA_GENERATOR_PATH"] = LORA_GENERATOR_PATH
if DISABLE_LORA:
    env_overrides["DISABLE_LORA"] = str(DISABLE_LORA)
if JITTER_BUFFER_SEC:
    env_overrides["JITTER_BUFFER_SEC"] = str(JITTER_BUFFER_SEC)


if EMOTION != "none" and EMOTION_CHECKPOINT:
    env_overrides["EMOTION_CHECKPOINT"] = EMOTION_CHECKPOINT
    env_overrides["EMOTION_ID"] = str(EMOTION_ID)


print("Resolved environment overrides for run_live.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={v}")


## Step 14 — Launch the live server

Runs `bash run_live.sh` in the background (it sources the venv itself), logs
to `live_server.log`, and records the PID for the stop/recovery cells below.


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", "run_live.sh"], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 15 — Wait for healthy startup

Polls the log for the documented healthy markers and the listening port
simultaneously. On timeout or early process exit, it dumps the log tail and
common-failure hints (CUDA OOM, missing path, traceback) instead of hanging
forever.


In [ ]:
HEALTHY_MARKERS = {
    "lora loaded": "lora loaded",
    "using direct Moshi reply hidden": "using direct Moshi reply hidden",
    "installed PersonaPlex graphed hidden capture": "installed PersonaPlex graphed hidden capture",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
    "serving static html": "index_v3_binary_fullscreen_aj_nodrop.html",
}

if EMOTION != "none" and EMOTION_CHECKPOINT:
    HEALTHY_MARKERS["emotion loaded"] = "emotion loaded"


ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "No such file or directory": "a referenced checkpoint/asset path is wrong — re-run Step 10's asset table",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 2 (nvidia-smi) and the torch CUDA build",
    "Address already in use": "the port was taken by another process after the Step 12 check — re-run Step 12",
    "Cannot find": "the VOICE_PROMPT file was not found under VOICE_PROMPT_DIR — re-run Step 10 to fetch/extract voices.tgz",
    "Missing required path": "start_winner_live.sh could not find a required checkpoint/prompt/HTML path — re-run Step 10 and check its printed list",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(120)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()


## Step 16 — HTTP / Uvicorn confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:{PORT}` (default `8998`), not just that the port is open. This only
checks `127.0.0.1` inside the pod — it does **not** prove the RunPod public
proxy URL works, since that depends on `PORT` being one of the ports actually
exposed by your pod template (see the note in the Parameters cell). This cell
prints the expected public URL from `RUNPOD_POD_ID` + `PORT` for convenience.


In [ ]:
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        body = resp.read(2000).decode(errors="ignore")
        print(f"[ok] HTTP {resp.status} from http://127.0.0.1:{PORT}/")
        print(body[:500])
except urllib.error.URLError as e:
    tail_log(80)
    raise RuntimeError(f"HTTP check failed against http://127.0.0.1:{PORT}/: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker live server (AH AudioPace) is running and healthy")
print(f"  Internal: http://{HOST}:{PORT}")
if PUBLIC_URL:
    print(f"  Public:   {PUBLIC_URL}")
    print(f"  (auto-built from RUNPOD_POD_ID={RUNPOD_POD_ID} and PORT={PORT} — confirm this exact")
    print(f"   port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard;")
    print(f"   the proxy only forwards ports that were exposed when the pod was created/edited)")
else:
    print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access the browser UI")
    print(f"  (RUNPOD_POD_ID was not auto-detected; set it in the Parameters cell for a printed URL)")
print(f"  Browser UI file: static/index_v3_binary_fullscreen_aj_nodrop.html")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Required assets ==")
    for asset in REQUIRED_ASSETS:
        print(f"  [{'OK' if os.path.exists(asset['path']) else 'MISSING'}] {asset['path']}")
    print(f"  [{'OK' if os.path.exists(VOICE_PROMPT_PATH) else 'MISSING'}] {VOICE_PROMPT_PATH}")

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["kill", "-9", pid])
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")
